# 00 · 환경 설정과 실행 준비

**2026-11-05 업데이트:** CUDA 그래프 탐색 + GPU 피처 캐시, 논리 배치 8,192 / 마이크로배치 최대 4,096.
원본에서 전처리하고 새로 학습합니다. 실행 코드는 **6개 노트북 안에 모두 포함**되어 있습니다.
Git 공유 설명은 [README](README.md), 변경 근거·논문 비교·HGB 계획은 [방법 문서](속도개선_학습방식_논문비교_HGB계획.md)를 확인하세요.

| 순서 | 하는 일 |
|---|---|
| 00 | 경로·배치 설정, 노트북의 실행 소스 생성 |
| 01 | 원본 전처리·거래 피처·일별 계좌 피처 |
| 02 | CSR·전체 모집단 준비, 선택한 CPU/CUDA 경로 검증 |
| 03 | A3 학습 → A3 Val·고정·Test → A0·A1·A2 학습·평가 |
| 04 | A3 등 개별 모델 평가 재개, 전체 평가 재개, 결과 열람 |
| 05 | 작은 합성 원본으로 전처리부터 4개 모델 학습·평가까지 CPU 검증 |

처음에는 **00 → 05**로 확인한 뒤 **01 → 02 → 03**을 실행합니다. 04는 결과 확인이나 개별 평가 재개에 사용합니다.
큰 작업은 `RUN_...` / `START_...`를 켜야 시작합니다. 원본은 `HI-Large_Trans.csv`, `HI-Large_accounts.csv`, `HI-Large_Patterns.txt`입니다.
데이터·체크포인트는 Git/ZIP에 포함하지 않습니다. Linux/WSL2, Python 3.12, GCC를 사용합니다.
CUDA 경로는 CUDA PyTorch와 그에 포함된 NVRTC를 사용하며, GPU 기본값의 측정 환경은 V100 32GB입니다.
원본 전처리·CSR 인덱스·일별 degree 캐시 생성은 CPU/디스크에서 수행하고, **반복되는 배치 그래프 탐색·피처 조합·학습을 GPU에서 처리**합니다.
원본을 새로 처리한 캐시가 기존 실험 캐시와 완전히 같다는 검증은 없으므로, 기존 가중치를 자동으로 이어 쓰지 않습니다.


In [ ]:
import sys
import subprocess

INSTALL_DEPENDENCIES = False
TORCH_INDEX = "https://download.pytorch.org/whl/cu126"  # CPU 전용은 끝의 cu126을 cpu로 변경
if INSTALL_DEPENDENCIES:
    subprocess.run([sys.executable, "-m", "pip", "install", "torch==2.10.0", "--index-url", TORCH_INDEX], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "numpy==2.4.1", "pandas==3.0.3", "pyarrow==24.0.0", "duckdb==1.5.5", "torch-geometric==2.8.0.post1", "ipykernel"], check=True)
else:
    print("설치 생략. 패키지를 새로 설치했다면 커널을 재시작하세요.")


In [ ]:
from pathlib import Path
import os
import json
import shutil

PROJECT_DIR = Path.cwd().resolve()  # 6개 ipynb가 있는 폴더
RAW_DIR = (PROJECT_DIR.parent.parent / "IBM").resolve()  # 필요하면 원본 폴더로 수정
WORK_DIR = (PROJECT_DIR / "work").resolve()              # 충분한 저장 공간이 있는 새 폴더
DEVICE = "cuda"  # GPU가 없으면 "cpu"
CUDA_MEMORY_FRACTION = 0.9
LOGICAL_BATCH = 8192
MICROBATCH_MAX = 4096
MAX_EDGES_PER_MICROBATCH = 6_000_000
DUCKDB_MEMORY_LIMIT = "6GB"
DUCKDB_THREADS = 2

assert sys.platform.startswith("linux"), "Linux/WSL2에서 실행하세요."
assert shutil.which("gcc"), "GCC를 설치한 뒤 실행하세요."
assert DEVICE in ("cpu", "cuda") and 0 < CUDA_MEMORY_FRACTION <= 1
assert 0 < MICROBATCH_MAX <= LOGICAL_BATCH and MAX_EDGES_PER_MICROBATCH > 0
assert WORK_DIR != RAW_DIR and not WORK_DIR.is_relative_to(RAW_DIR), "출력은 원본 폴더 밖에 지정하세요."
RAW_FILES = {
    "transactions": RAW_DIR / "HI-Large_Trans.csv",
    "accounts": RAW_DIR / "HI-Large_accounts.csv",
    "patterns": RAW_DIR / "HI-Large_Patterns.txt",
}
for name, path in RAW_FILES.items():
    print(name, path, "존재" if path.is_file() else "경로 확인 필요")
print("출력:", WORK_DIR)


## 공통 소스

각 단계의 코드가 해당 노트북 안에 들어 있습니다. 다음 생성 셀은 6개 노트북의 `SOURCES[...]`에 등록된
문자열만 읽어 Python·C 실행 파일을 만듭니다. 다른 노트북의 학습 셀을 실행하지 않습니다.

### 이 단계의 실행 코드

아래 소스는 00 노트북이 읽어 작업 폴더에 실행 파일로 만듭니다. 코드를 수정하면 노트북을 저장하고 **새 WORK_DIR**로 00을 다시 실행하세요.

In [ ]:
SOURCES = {}


In [ ]:
# salt_daily/__init__.py
SOURCES['salt_daily/__init__.py'] = '''"""Isolated daily-close SALT experiments; never modifies the FraudGT run."""

'''


In [ ]:
# runtime/__init__.py
SOURCES['runtime/__init__.py'] = ''''''


In [ ]:
# salt_daily/config.py
SOURCES['salt_daily/config.py'] = '''"""Paths configured by the notebook; all generated files stay under ROOT."""
import json
import os
from pathlib import Path

ROOT = Path(__file__).resolve().parents[1]
settings_path = ROOT / 'local_settings.json'
settings = json.loads(settings_path.read_text()) if settings_path.exists() else {}
DATA_ROOT = Path(settings.get('data_root', ROOT / 'data')).expanduser().resolve()
def input_path(name):
    value = Path(settings.get('paths', {}).get(name, DATA_ROOT / name)).expanduser()
    return value.resolve() if value.is_absolute() else (DATA_ROOT / value).resolve()
FEATURES = input_path('features')
GRAPH = input_path('graph')
STRUCTURAL = input_path('structural')
PORTS = input_path('ports')
FRAUDGT = input_path('labels')
RUN = ROOT/'runs/daily_close_001'
LEGACY = ROOT  # Retained import compatibility; no legacy runtime dependency.
for name in ('OMP_NUM_THREADS', 'MKL_NUM_THREADS', 'OPENBLAS_NUM_THREADS'):
    os.environ.setdefault(name, '1')
'''


In [ ]:
# salt_daily/storage.py
SOURCES['salt_daily/storage.py'] = '''from datetime import datetime, timezone
import hashlib
import json
import os
from pathlib import Path


def utc():
    return datetime.now(timezone.utc).isoformat()


def read_json(path):
    return json.loads(Path(path).read_text())


def write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + f'.{os.getpid()}.tmp')
    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False) + '\\n')
    temporary.replace(path)


def sha(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda: handle.read(4 * 1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()


def log(folder, event, **values):
    row = dict(utc=utc(), event=event, **values)
    folder.mkdir(parents=True, exist_ok=True)
    with (folder / 'events.jsonl').open('a') as handle:
        handle.write(json.dumps(row, ensure_ascii=False, allow_nan=False) + '\\n')
    write_json(folder / 'status.json', row)
    print(json.dumps(row, ensure_ascii=False), flush=True)


def memory():
    """Linux telemetry; cgroup v2 is optional on a teammate's host."""
    values = {}
    status = Path('/proc/self/status')
    if status.exists():
        for line in status.read_text().splitlines():
            if line.split(':')[0] in ('VmRSS', 'RssAnon', 'RssFile'):
                key, value = line.split(':', 1)
                values[key + '_bytes'] = int(value.split()[0]) * 1024
    for name in ('memory.current', 'memory.max'):
        path = Path('/sys/fs/cgroup') / name
        value = path.read_text().strip() if path.exists() else 'unavailable'
        values[name] = int(value) if value.isdigit() else value
    return values
'''


In [ ]:
# salt_daily/portable.py
SOURCES['salt_daily/portable.py'] = '''"""Verify the dataset identity, input file states and runtime source identity."""
from .config import ROOT, DATA_ROOT
from .storage import read_json, sha


def input_files():
    for item in read_json(DATA_ROOT/'manifest.json')['files']:
        yield item, DATA_ROOT/item['path']


def check_inputs():
    import numpy as np
    manifest = read_json(DATA_ROOT/'manifest.json')
    protocol = read_json(ROOT/'protocol.json')
    assert protocol['dataset_id'] == manifest['dataset_id']
    assert protocol['input_manifest_sha256'] == sha(DATA_ROOT/'manifest.json')
    assert sha(ROOT/'calibration.json') == manifest['calibration_sha256']
    stats = {}
    for item, path in input_files():
        s = path.stat()
        if s.st_size != item['bytes'] or s.st_mtime_ns != item['mtime_ns']:
            raise ValueError(f'Prepared input changed: {path}')
        if path.suffix == '.npy':
            a = np.load(path, mmap_mode='r', allow_pickle=False)
            if a.dtype.str != item['dtype'] or list(a.shape) != item['shape'] or not a.flags.c_contiguous:
                raise ValueError(f'Prepared array schema changed: {path}')
            del a
        stats[str(path)] = dict(size=s.st_size,mtime_ns=s.st_mtime_ns)
    return stats


def code_manifest():
    paths=[]
    for directory in ('salt_daily','runtime','tests'):
        paths.extend(p for p in (ROOT/directory).glob('*') if p.suffix in ('.py','.c','.cu'))
    paths.extend(ROOT/name for name in ('run.py','notebook_control.py','local_settings.json',
                                      'raw_settings.json','calibration.json','docs/input_contract.json'))
    return {str(p.relative_to(ROOT)):sha(p) for p in sorted(paths)}
'''


In [ ]:
# salt_daily/raw_config.py
SOURCES['salt_daily/raw_config.py'] = '''"""Raw-data settings shared by the preprocessing notebooks."""
from dataclasses import dataclass

ENTITY_TYPES = ('Corporation','Partnership','Sole Proprietorship','Individual','Country','Direct')
FORMAT_NAMES = ('ACH','Bitcoin','Cash','Cheque','Credit Card','Reinvestment','Wire','Unknown')
PATTERN_NAMES = ('NORMAL','BIPARTITE','CYCLE','FAN-IN','FAN-OUT','GATHER-SCATTER','RANDOM','SCATTER-GATHER','STACK','UNMATCHED_POSITIVE')

@dataclass(frozen=True)
class RawConfig:
    start: str = '2022-08-01'
    end: str = '2022-11-01'
    calibration_end: str = '2022-08-08'
    train_end: str = '2022-09-26'
    memory_limit: str = '6GB'
    threads: int = 2
    chunk_rows: int = 250_000
    require_hi_large_counts: bool = True

    def __post_init__(self):
        if (self.start,self.end,self.calibration_end,self.train_end) != ('2022-08-01','2022-11-01','2022-08-08','2022-09-26'):
            raise ValueError('This SALT adapter fixes the original 2022-08..10 experiment dates')
        if self.threads < 1 or self.chunk_rows < 1:
            raise ValueError('threads and chunk_rows must be positive')
'''


In [ ]:
# run.py
SOURCES['run.py'] = '''"""Portable pipeline: A3 Train/Monitor -> Val/freeze/Test -> other models."""
import argparse
import fcntl
import os
from salt_daily.config import ROOT, RUN
from salt_daily.storage import read_json, write_json, sha, log, utc


def verify_ready():
    from salt_daily.portable import check_inputs, code_manifest
    protocol = read_json(ROOT/'protocol.json')
    result = read_json(ROOT/'preflight.json')
    manifest = read_json(ROOT/'code_manifest.json')
    assert result['status'] == 'pass' and result['protocol_sha256'] == sha(ROOT/'protocol.json')
    assert result['code_manifest'] == manifest == code_manifest(), 'Code/settings changed; use a new WORK_DIR'
    assert result['input_file_stats'] == check_inputs(), 'Input cache changed since validation'
    assert sha(ROOT/'cache/population/population.json') == protocol['population_manifest_sha256']
    for item in protocol['cohorts'].values():
        for name, digest in item['files'].items():
            assert sha(ROOT/'cache/population'/name) == digest, f'Population changed: {name}'
    return protocol


def lock_held(path):
    if not path.exists():
        return False
    with path.open('r+') as handle:
        try:
            fcntl.flock(handle, fcntl.LOCK_EX | fcntl.LOCK_NB)
            fcntl.flock(handle, fcntl.LOCK_UN)
            return False
        except BlockingIOError:
            return True


def check_device(device):
    import torch
    if device == 'cuda':
        if not torch.cuda.is_available():
            raise RuntimeError('CUDA unavailable; install CUDA PyTorch or choose CPU')
        if read_json(ROOT/'preflight.json')['device'] != 'cuda':
            raise RuntimeError('Run validate --device cuda before CUDA training/evaluation')


def run(device, phase="all"):
    RUN.mkdir(parents=True, exist_ok=True)
    with (RUN/'run.lock').open('a') as handle:
        fcntl.flock(handle, fcntl.LOCK_EX | fcntl.LOCK_NB)
        protocol = verify_ready()
        if (RUN/'complete.json').exists():
            assert read_json(RUN/'complete.json')['protocol_sha256'] == sha(ROOT/'protocol.json')
            print('Full experiment already complete')
            return
        check_device(device)
        log(RUN, 'job_started', pid=os.getpid(), manual_launch=True, population_ipw=False)
        from salt_daily.data import DailyData
        loader = DailyData(device=device)
        run_pipeline(protocol, loader, device, phase)


def run_pipeline(protocol, loader, device, phase='all'):
    from salt_daily.training import train_variant, evaluate_scope, freeze_selection, evaluate_variant, require_trained_variant
    if phase not in ('all', 'train', 'evaluate'):
        raise ValueError(phase)
    early = protocol['evaluation'].get('independent_variants', [])
    assert set(early).issubset(protocol['variants'])
    if phase in ('all', 'train'):
        for variant in protocol['variants']:
            log(RUN, 'variant_started', variant=variant, pid=os.getpid())
            train_variant(variant, protocol, loader, device)
            if phase == 'all' and variant in early:
                log(RUN, 'variant_final_evaluation_started', variant=variant, before_other_models=True)
                evaluate_variant(protocol, loader, variant, device)
    if phase == 'train':
        log(RUN, 'all_training_complete', next_step='04 evaluation notebook')
        return
    for variant in protocol['variants']:
        require_trained_variant(protocol, variant)
    validation = evaluate_scope(protocol, loader, 'validation', device)
    selection = freeze_selection(protocol, validation)
    test = evaluate_scope(protocol, loader, 'test', device)
    write_json(RUN/'complete.json', dict(status='complete', completed_utc=utc(),
        protocol_sha256=sha(ROOT/'protocol.json'), selection=selection, validation=validation,
        test=test, independently_evaluated_variants=early,
        full_population_certified=True, population_ipw=False))
    log(RUN, 'experiment_complete', full_population_certified=True, test_evaluated=True)


def evaluate_one(variant, device):
    from salt_daily.training import require_trained_variant, evaluate_variant
    RUN.mkdir(parents=True, exist_ok=True)
    with (RUN/'run.lock').open('a') as handle:
        fcntl.flock(handle, fcntl.LOCK_EX | fcntl.LOCK_NB)
        protocol = verify_ready()
        require_trained_variant(protocol, variant)
        check_device(device)
        from salt_daily.data import DailyData
        return evaluate_variant(protocol, DailyData(device=device), variant, device)


if __name__ == '__main__':
    parser = argparse.ArgumentParser()
    parser.add_argument('command', choices=('ingest', 'features', 'nodes', 'graph', 'prepare', 'validate', 'train', 'evaluate', 'run', 'status', 'check'))
    parser.add_argument('--device', choices=('cpu', 'cuda'), default='cpu')
    parser.add_argument('--variant', choices=('A3', 'A0', 'A1', 'A2'), default=None)
    args = parser.parse_args()
    try:
        if args.command in ('ingest', 'features', 'nodes'):
            from salt_daily.preprocessing import ingest_raw, build_edges, build_daily_nodes
            {'ingest':ingest_raw,'features':build_edges,'nodes':build_daily_nodes}[args.command]()
        elif args.command in ('graph', 'prepare'):
            from salt_daily.graph_prepare import build_graph, prepare_inputs
            {'graph':build_graph,'prepare':prepare_inputs}[args.command]()
        elif args.command == 'validate':
            RUN.mkdir(parents=True, exist_ok=True)
            with (RUN/'run.lock').open('a') as handle:
                fcntl.flock(handle, fcntl.LOCK_EX | fcntl.LOCK_NB)
                from salt_daily.validate import validate
                print(validate(args.device)['status'])
        elif args.command in ('run', 'train', 'evaluate'):
            if args.command == 'evaluate' and args.variant:
                evaluate_one(args.variant, args.device)
            else:
                run(args.device, {'run':'all','train':'train','evaluate':'evaluate'}[args.command])
        elif args.command == 'check':
            from salt_daily.portable import check_inputs
            print(f'Prepared cache: {len(check_inputs())} files checked')
        else:
            print((RUN/'status.json').read_text() if (RUN/'status.json').exists() else 'Paused; not started')
    except BaseException as error:
        if args.command in ('run', 'train', 'evaluate') and not isinstance(error, BlockingIOError):
            log(RUN, 'job_stopped', error_type=type(error).__name__, detail=str(error)[:1000])
        raise
'''


In [ ]:
# docs/input_contract.json
SOURCES['docs/input_contract.json'] = '''{
  "version": "v10-daily-close-all-minute-edges-v2",
  "decision_time": "day close; timestamp < next midnight",
  "node_features": [
    "log_out_count",
    "log_in_count",
    "log_out_neighbors",
    "log_in_neighbors",
    "log_self_count",
    "log_reciprocal_neighbors",
    "mean_paid_currency_z",
    "paid_outlier_fraction",
    "log_currency_count",
    "log_active_days",
    "type_Corporation",
    "type_Partnership",
    "type_Sole Proprietorship",
    "type_Individual",
    "type_Country",
    "type_Direct",
    "log_entity_observed_accounts",
    "log_entity_out_diversity",
    "log_entity_in_diversity"
  ],
  "edge_features": [
    "pair_prior_count",
    "pair_gap_minutes",
    "pair_count_1h",
    "sender_gap_minutes",
    "sender_count_1h",
    "receiver_gap_minutes",
    "receiver_count_1h",
    "paid_currency_z",
    "pair_history_count",
    "reverse_history_count",
    "pair_age_at_day_start",
    "same_entity",
    "same_known_country",
    "both_countries_known",
    "same_currency",
    "is_self_transfer",
    "amt_z_pay_ccy"
  ],
  "combiner_features": [
    "payment_format",
    "src_log_entity_observed_accounts",
    "minute_receiver_vs_prior_hour",
    "minute_sender_vs_prior_hour",
    "receiver_gap_minutes",
    "minute_log_src_out_peers",
    "minute_log_dst_in_peers"
  ],
  "node_feature_types": 19,
  "edge_feature_types": 17,
  "new_unique_combiner_types": 5,
  "total_v10_candidate_data_feature_types": 41,
  "added_by_this_preprocessing_fix": 0,
  "gnn_ego_indicator": "additional structural mask, not a transaction feature",
  "graph_policy": "Every target-minute incident edge at each of two hops; up to 100 other-history edges per node/direction",
  "feature_dtype": "float32",
  "label_allowed_only_as_supervision": true,
  "pattern_metadata_allowed_in_context": false,
  "sequence_policy": "Compatibility mode: 32 strictly-prior role-events per endpoint, not 32 unique transactions",
  "sequence_numeric_token_features": 8,
  "sequence_target_numeric_features": 5,
  "sequence_format_embedding": true,
  "sequence_fusion_columns": 10,
  "sequence_columns_are_not_total_raw_feature_count": true,
  "train_transform_window": [
    "2022-08-01",
    "2022-08-07"
  ],
  "static_kyc_asof_availability": "not verifiable from current source",
  "original_split_end_cache_reuse_allowed": false
}
'''


In [ ]:
# protocol.template.json
SOURCES['protocol.template.json'] = '''{
  "version": "salt-daily-raw-notebooks-cuda-v2",
  "splits": {
    "calibration": [
      "2022-08-01",
      "2022-08-07"
    ],
    "train": [
      "2022-08-08",
      "2022-09-25"
    ],
    "monitor": [
      "2022-09-26",
      "2022-10-04"
    ],
    "fusion_fit": [
      "2022-10-05",
      "2022-10-08"
    ],
    "validation": [
      "2022-10-09",
      "2022-10-13"
    ],
    "test": [
      "2022-10-14",
      "2022-10-31"
    ]
  },
  "inputs": {
    "node_features": [
      "log_out_count",
      "log_in_count",
      "log_out_neighbors",
      "log_in_neighbors",
      "log_self_count",
      "log_reciprocal_neighbors",
      "mean_paid_currency_z",
      "paid_outlier_fraction",
      "log_currency_count",
      "log_active_days",
      "type_Corporation",
      "type_Partnership",
      "type_Sole Proprietorship",
      "type_Individual",
      "type_Country",
      "type_Direct",
      "log_entity_observed_accounts",
      "log_entity_out_diversity",
      "log_entity_in_diversity"
    ],
    "edge_features": [
      "pair_prior_count",
      "pair_gap_minutes",
      "pair_count_1h",
      "sender_gap_minutes",
      "sender_count_1h",
      "receiver_gap_minutes",
      "receiver_count_1h",
      "paid_currency_z",
      "reverse_history_count",
      "pair_age_at_day_start",
      "same_known_country",
      "both_countries_known",
      "same_currency",
      "is_self_transfer"
    ],
    "business_features": 34,
    "ego_indicator": true,
    "payment_format_input": true,
    "payment_format_names": [
      "ACH",
      "Bitcoin",
      "Cash",
      "Cheque",
      "Credit Card",
      "Reinvestment",
      "Wire",
      "Unknown"
    ],
    "payment_format_encoding": "one-hot, 8 columns; invalid codes use Unknown",
    "port_numbering": true,
    "structural_features": [
      "in_port",
      "out_port"
    ],
    "port_encoding": "raw float32, zero-based first distinct counterpart order by (minute, edge_id); immutable historical ranks",
    "edge_tensor_columns": 24
  },
  "cohorts": {},
  "graph": {
    "batch_mode": "shared union of per-query sampled graphs within each GPU microbatch",
    "context": "all observed transactions through target day close",
    "directions": [
      "incoming",
      "outgoing"
    ],
    "hops": 2,
    "other_minute_cap_per_node_direction": 100,
    "all_target_minute_edges_at_each_hop": true,
    "sampler": "CUDA two-hop expansion of unique (account, query-minute) states, same priority CSR and exact CPU graph union for identical target batches",
    "parallel_transactions": "retain distinct edge IDs",
    "labels_in_context": false,
    "graph_independent_of_other_supervised_queries_required": false,
    "ego_scope": "all supervised transaction endpoints in the shared microbatch",
    "direction_flags": "bitwise OR across occurrences of the same transaction ID",
    "post_union_fanout": "may exceed 100 per node/direction; cap100 applies to original per-query selection before union",
    "target_batch_size": 4096,
    "batch_membership_affects_prediction": true,
    "sampler_backend": "cuda"
  },
  "degree": {
    "primary_scaling_degree": "exact observed original transaction count per direction through target day close",
    "statistics_over": "actually sampled messages",
    "track": [
      "context_degree",
      "sampled_degree",
      "unique_counterparties"
    ],
    "delta_window": [
      "2022-08-01",
      "2022-09-25"
    ],
    "delta_snapshot": "2022-09-25 training graph close",
    "delta_population": "active account union in training graph; include zero degree per direction",
    "delta_definition": "separate mean(log1p(in_degree)) and mean(log1p(out_degree)); freeze",
    "delta_future_nodes_excluded": "Monitor/Val/Test-only nodes excluded; full Train statistics fitted once",
    "synthetic_reverse_edges_in_original_degree": false,
    "zero_degree_guard_required": true,
    "project_choice_not_paper_reproduction": true
  },
  "architecture": {
    "common_hidden": 20,
    "heads": 2,
    "layers": 2,
    "activation": "ReLU",
    "dropout": 0.083,
    "common_input_projection": true,
    "common_target_head": true,
    "all_query_targets_scored": true,
    "statistics": [
      "mean",
      "population_std",
      "min",
      "max"
    ],
    "scalers": [
      "identity",
      "amplification",
      "attenuation"
    ],
    "fusion": "concat(previous,statistical,attention), MLP, average residual; feed fused state into next layer",
    "models": {
      "A0": "attention_only",
      "A1": "statistics_only",
      "A2": "prediction_time_fusion",
      "A3": "layerwise_salt_trans"
    },
    "common_A0_is_not_claimed_identical_to_frozen_V10": true,
    "parameter_equality_not_assumed": true,
    "implementation_contract_to_freeze_before_training": [
      "direction merge and root path",
      "projection dimensions and MLP layers",
      "normalization and exact residual placements in every control",
      "edge conditioning and parallel-edge handling",
      "self-loop policy and zero-degree branch",
      "common head input width and target index mapping"
    ],
    "followup_controls": [
      "attention with explicit same context-degree at each layer",
      "attention parameter count within 5% of SALT",
      "SALT without degree scalers",
      "SALT using mean+std only"
    ],
    "implementation_contract": {
      "hidden": 20,
      "heads": 2,
      "layers": 2,
      "node_projection": "Linear20to20 + ReLU",
      "attention": "per-direction TransformerConv20 with 2 heads, edge_dim24, root_weight=False, bias=False, coefficient dropout0",
      "unused_pyg_skip_projection": "frozen; excluded from trainable parameter counts",
      "statistics": "h_source + Linear(edge24to20); population mean/std/min/max; 3 degree scalers; concat h_destination; MLP260to20to20",
      "direction_merge": "separate Linear40to20 for attention and statistics",
      "fusion": "MLP concat(previous, available branches) to20to20; (previous+fusion)/2, then Dropout0.083",
      "A2": "independent A0 and A1 towers, final MLP concat(initial,statistics,attention), residual average with initial",
      "head": "Linear64to20, ReLU, Dropout0.288, Linear20to2; score=logit1-logit0",
      "normalization": "none beyond structural degree scaling",
      "self_loops": "no synthetic self loops; observed self transfers retained",
      "std": "population centered second moment; guarded sqrt and zero std for zero variance",
      "original_degree": "unscaled exact observed count, through query day close; sampled degrees separate"
    },
    "final_dropout": 0.288,
    "output_classes": 2,
    "dropout_placement": "hidden states after residual fusion; final hidden layer before two-class head",
    "attention_coefficient_dropout": 0.0,
    "edge_dim": 24
  },
  "training": {
    "seed": 42,
    "optimizer": "Adam",
    "loss": "two-class weighted cross entropy; PyTorch weighted mean over logical batch",
    "positive_weight": 7,
    "sampling_weight_normalization": "none",
    "full_train_population": 87572226,
    "logical_batch": 8192,
    "microbatch_max": 4096,
    "max_edges_per_microbatch": 6000000,
    "gradient_clip": 5,
    "epochs_primary": 6,
    "primary_early_stopping": false,
    "learning_rates_primary": [
      0.0012,
      0.0012,
      0.0012,
      0.0012,
      0.0012,
      0.0012
    ],
    "targets_per_epoch": 87572226,
    "targets_per_six_epochs": 525433356,
    "checkpoint": "maximum unweighted full Monitor F1 at fixed probability >=0.5; earliest tie",
    "seeds_final": [
      42,
      0,
      1
    ],
    "representative_seed": 42,
    "recipe_source": "SALT-Trans final hyperparameters; full targets, no IPW, daily-close HI-Large adapter",
    "day_order": "ascending every epoch",
    "day_boundary_batch_policy": "flush final shorter logical batch; never mix days",
    "epochs_note": "6 is the retained initial pilot budget, not a paper epoch count; no claimed convergence",
    "target_sampling": "none; all observed transactions in the Train date range",
    "seeds_final_status": "planned only; initial run uses seed42",
    "population_ipw": false,
    "checkpoint_resume": "Resume only within identical generated code, inputs and protocol; fresh raw data requires a new run.",
    "updates_per_full_epoch": 10714,
    "gpu_batch_tuning": "Defaults transferred from the source V100 32GB Train-only benchmark; validate on your generated cache and hardware.",
    "cuda_memory_fraction": 0.9
  },
  "variants": [
    "A3",
    "A0",
    "A1",
    "A2"
  ],
  "evaluation": {
    "monitor": "all observed Monitor transactions; no population/sample weights",
    "candidate_selection": "full held-out Val F1; checkpoint selected on full Monitor only",
    "full_validation_before_goal_claim": true,
    "full_test_after_freeze": true,
    "full_validation_rows": 8551015,
    "full_historical_test_rows": 33429507,
    "coverage": 1,
    "no_test_tuning": true,
    "no_best_seed_cherry_pick": true,
    "degree_bins": [
      [
        0,
        0
      ],
      [
        1,
        4
      ],
      [
        5,
        9
      ],
      [
        10,
        19
      ],
      [
        20,
        49
      ],
      [
        50,
        99
      ],
      [
        100,
        199
      ],
      [
        200,
        499
      ],
      [
        500,
        999
      ],
      [
        1000,
        null
      ]
    ],
    "metrics": [
      "precision",
      "recall",
      "F1",
      "AP",
      "known_pattern_recall",
      "unmatched_recall",
      "TP",
      "FP",
      "FN",
      "TN",
      "coverage",
      "top_k"
    ],
    "one_global_threshold_for_all_subgroups": true,
    "threshold_probability": 0.5,
    "threshold_logit_difference": 0.0,
    "threshold_search": false,
    "checkpoint_selection_split": "monitor",
    "validation_role": "full held-out comparison after Monitor checkpoint selection",
    "degree_bins_status": "planned diagnostic; not emitted in this settings change",
    "attention_diagnostics_status": "ENN/Hit@k not implemented",
    "population_ipw": false,
    "score_commit_targets": 8192,
    "test_policy": "A3: completed Train/Monitor -> full Val -> per-variant checkpoint/Val freeze -> full Test, before A0 training. Others: full Val comparison and global freeze before Test. Reuse completed A3 Test; never select or tune using Test.",
    "top_k_fractions": [
      0.001,
      0.005,
      0.01
    ],
    "thresholds": [
      0.1,
      0.2,
      0.3,
      0.4,
      0.5,
      0.6,
      0.7,
      0.8,
      0.9,
      1.0
    ],
    "independent_variants": [
      "A3"
    ],
    "independent_validation_role": "A3 full Val is reported and locked with its Monitor-selected checkpoint; global model comparison still uses full Val only after other models finish"
  },
  "limitations": [
    "Custom HI-Large inputs and daily-close adapter; not an exact HI-Small/Medium paper reproduction",
    "Same-day later transactions may appear in the graph; D+1 and later excluded",
    "Sampled message statistics do not equal full-neighborhood statistics",
    "Single seed and six full-population epochs retained as initial budget, not a paper convergence claim",
    "Bidirectional towers, heads=2 and dropout placement remain project choices",
    "Degree normalization fitted on the full Train graph and frozen",
    "Attention ENN/Hit@k diagnostic remains pending"
  ],
  "execution": {
    "state": "fresh_notebook_run",
    "automatic_start": false,
    "reuse_ipw_checkpoint": false,
    "pipeline": [
      "raw_ingest",
      "causal_edge_and_daily_node_features",
      "csr_and_population",
      "validate_inputs",
      "train_with_full_monitor",
      "full_validation_and_frozen_test"
    ]
  },
  "stochastic_note": "Sharing changes node neighborhoods, batch-wide Ego and dropout assignment; model outputs are not expected to equal disjoint outputs. Batch composition is part of the evaluation protocol.",
  "packaging": {
    "kind": "staged notebooks from raw CSV",
    "fresh_training_default": true,
    "included_weights": false
  },
  "input_pipeline": {
    "backend": "cuda",
    "numeric_dtype": "float32; unchanged values",
    "categorical_storage": "four binary fields plus payment code and validity bit in uint8",
    "bytes_per_transaction": 49,
    "feature_cache_gib": 7.7792552430182695,
    "labels_cached": false,
    "daily_node_and_degree_refresh": true,
    "graph_and_features_stay_on_gpu": true,
    "split_before_feature_gather": true,
    "future_features_never_selected_before_daily_cutoff": true,
    "feature_cache_gib_note": "7.78 GiB measured on source HI-Large cache; 49 bytes times raw-ID width, not a fixed allocation."
  },
  "handoff_note": "Fresh preprocessing and fresh training; source A3 first epoch used mixed batch settings. No source checkpoint migration."
}
'''


In [ ]:
import ast
import hashlib

NOTEBOOKS = ['00_환경설정과_실행준비.ipynb', '01_원본전처리.ipynb', '02_그래프와_학습입력.ipynb', '03_SALT_GNN_모델학습.ipynb', '04_검증_Test_결과.ipynb', '05_전체흐름_스모크테스트.ipynb']
bundle = {}
for name in NOTEBOOKS:
    path = PROJECT_DIR / name
    if not path.is_file():
        raise FileNotFoundError(f"6개 노트북을 같은 폴더에 놓으세요: {name}")
    notebook = json.loads(path.read_text(encoding="utf-8"))
    for cell in notebook["cells"]:
        if "embedded-source" not in cell.get("metadata", {}).get("tags", []):
            continue
        source = cell["source"]
        source = "".join(source) if isinstance(source, list) else source
        tree = ast.parse(source)
        if len(tree.body) != 1 or not isinstance(tree.body[0], ast.Assign):
            raise ValueError("소스 셀은 SOURCES['파일명'] = '코드' 형태여야 합니다.")
        item = tree.body[0]
        target = item.targets[0]
        if not isinstance(target, ast.Subscript) or not isinstance(target.value, ast.Name) or target.value.id != "SOURCES":
            raise ValueError("잘못된 소스 셀")
        relative = ast.literal_eval(target.slice)
        content = ast.literal_eval(item.value)
        if not isinstance(relative, str) or not isinstance(content, str):
            raise ValueError("파일명과 소스는 문자열이어야 합니다.")
        relpath = Path(relative)
        if relpath.is_absolute() or ".." in relpath.parts or relative in bundle:
            raise ValueError(f"중복되거나 잘못된 경로: {relative}")
        bundle[relative] = content

def json_text(value):
    return json.dumps(value, ensure_ascii=False, indent=2) + "\n"

protocol = json.loads(bundle["protocol.template.json"])
protocol["training"].update(cuda_memory_fraction=CUDA_MEMORY_FRACTION,
    logical_batch=LOGICAL_BATCH, microbatch_max=MICROBATCH_MAX,
    max_edges_per_microbatch=MAX_EDGES_PER_MICROBATCH)
protocol["graph"].update(sampler_backend=DEVICE, target_batch_size=MICROBATCH_MAX)
protocol["input_pipeline"]["backend"] = DEVICE
bundle["protocol.template.json"] = json_text(protocol)
bundle["local_settings.json"] = json_text({"data_root":str(WORK_DIR/"data")})
bundle["raw_settings.json"] = json_text({"files":{k:str(v) for k,v in RAW_FILES.items()},
    "preprocess":{"memory_limit":DUCKDB_MEMORY_LIMIT,"threads":DUCKDB_THREADS,"require_hi_large_counts":True}})

# 먼저 모든 충돌을 확인한 후 생성합니다. 기존 실행의 코드·설정은 덮어쓰지 않습니다.
for relative, content in bundle.items():
    path = WORK_DIR / relative
    if path.exists() and path.read_text(encoding="utf-8") != content:
        raise RuntimeError(f"기존 실행 파일이 다릅니다: {path}. 새 WORK_DIR를 지정하세요.")
for relative, content in bundle.items():
    path = WORK_DIR / relative
    path.parent.mkdir(parents=True, exist_ok=True)
    if not path.exists():
        path.write_text(content, encoding="utf-8")
(PROJECT_DIR/".salt_project.json").write_text(json_text({"work_dir":str(WORK_DIR),"device":DEVICE}),encoding="utf-8")
print(f"실행 소스 {len(bundle)}개 준비 완료. 원본 전처리와 학습은 아직 시작하지 않았습니다.")


## 실행 조건과 재개

배치·GPU 메모리 비율은 위 설정 셀에서 정한 뒤 00을 실행하고 02에서 검증합니다. 부족한 GPU에서는 배치를 줄이고 **새 WORK_DIR**에서 검증하세요.
배치 구성은 공유 그래프와 Ego 표시를 바꾸므로 설정이 달라지면 학습 궤적과 예측도 달라질 수 있습니다.
원본 경로·생성 소스·설정이 달라진 기존 WORK_DIR는 덮어쓰지 않습니다. 동일 조건의 중단 재개만 지원합니다.
CPU 모드는 기능 확인용입니다. 05의 스모크는 실제 원본과 분리하여 논리 16 / micro 8, 1에폭으로 실행합니다.

날짜 구간: Train 08-08~09-25 / Monitor 09-26~10-04 / fusion_fit 예약 10-05~10-08 / Val 10-09~10-13 / Test 10-14~10-31 (2022년).
당일 마감 시점의 일괄 탐지이므로 같은 날 대상 거래보다 나중 거래를 참조할 수 있습니다. 거래 발생 즉시 탐지와 구분하세요.
HGB 점수 결합은 후속 계획이며 이 노트북의 기본 학습에는 포함되지 않습니다.
